In [36]:
# Imports

import numpy as np
import pandas as pd
from shared.utils import load_data
from datasets import preprocess_dataset, datasets_types

from sklearn.cluster import KMeans
from scipy.stats import beta as beta_dist
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report
from art.attacks.evasion import HopSkipJump

from art.estimators.classification.scikitlearn import ScikitlearnClassifier
from art.utils import to_categorical
from art.attacks.evasion import ZooAttack
from art.estimators.classification import SklearnClassifier
seed = 42

from attacks.attack_utils import (apollon_classifier, evaluate, functional_indexes,
                                  model_classifier, restore_functional, run_attack)
from MAB.apollon_mab import ApollonMAB

In [3]:
df = load_data(
            [
                "./shared/data/CIC_2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
                "./shared/data/CIC_2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
                "./shared/data/CIC_2017/Tuesday-WorkingHours.pcap_ISCX.csv"
            ]
        )
print("Dataset cargado")
df_preprocessed = preprocess_dataset(
    df, save=True, dataset_type="CIC_2017", seed=seed, load=False)
print("Dataset Preprocesado")

Dataset cargado
Loading new data
labels: {'-', 'FTP-Patator', 'SSH-Patator', 'Web Attack � XSS', 'Web Attack � Brute Force'}
Dataset Preprocesado


In [26]:
x_train = df_preprocessed.x_train
y_train = df_preprocessed.y_train
x_test = df_preprocessed.x_test
y_test = df_preprocessed.y_test

y_train = y_train.apply(lambda x: int(x))
y_test = y_test.apply(lambda x: int(x))

print(y_train.unique())
print(y_test.unique())

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

[0 1]
[0 1]
(633413, 63)
(633413,)
(271464, 63)
(271464,)


In [27]:
# Train a RF Classifier
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [28]:
# Train a MLP Classifier
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(32),max_iter=200,verbose=False,batch_size=200,early_stopping=True,activation='tanh',solver='adam')
mlp.fit(x_train, y_train)

MLPClassifier(activation='tanh', batch_size=200, early_stopping=True,
              hidden_layer_sizes=32)

In [29]:
# Train a NB Classifier
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(x_train, y_train)

GaussianNB()

In [30]:
# Train a DT Classifier
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)

DecisionTreeClassifier()

In [31]:
# Train a LR Classifier
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(x_train, y_train)

c:\Users\AntonioPayá\Desktop\Repos-Doctorado\apollon\venv\lib\site-packages\sklearn\linear_model\_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
# Functional features must keep their value for the attack to remain functional.
# The previous list was missing commas (19 entries instead of 27) and the indexes were taken
# from the raw CSV columns with swapped names. They are now computed on the columns that are
# actually fed to the models (see attacks/attack_utils.py).
FEATURE_COLUMNS = list(df_preprocessed.x.columns)
FUNCTIONAL_IDX, MODIFIABLE_IDX, MISSING = functional_indexes(FEATURE_COLUMNS)
print(len(FUNCTIONAL_IDX), "functional and", len(MODIFIABLE_IDX), "modifiable features")
print("Functional features not present after preprocessing:", MISSING)

In [ ]:
# The previous ART wrapper (meant to keep functional features fixed) was commented out and
# returned only the first prediction. Functional features are now restored after every attack
# with `restore_functional` (see attacks/attack_utils.py).

In [34]:
attacks_data = x_test[y_test == 1]
normal_data = x_test[y_test == 0]
print(attacks_data.shape, normal_data.shape)

(4815, 63) (266649, 63)


In [ ]:
ATTACK = "hsja"
N = 100
MAX_ITER = 100  # as stated in the paper (the previous calls used ART's default)
x_attack = np.asarray(attacks_data[:N])
x_normal = np.asarray(normal_data[:N])
y_eval = np.concatenate([np.ones(len(x_attack)), np.zeros(len(x_normal))])

# Adversarial examples crafted against each single model (targeted to the benign class).
models = {"RF": rf, "MLP": mlp, "NB": nb, "DT": dt, "LR": lr}
adv_by_model = {}
for name, model in models.items():
    x_adv = run_attack(ATTACK, model_classifier(model), x_attack, max_iter=MAX_ITER)
    x_adv = restore_functional(x_adv, x_attack, FUNCTIONAL_IDX)
    adv_by_model[name] = x_adv
    clean = evaluate(y_eval, model.predict(np.concatenate([x_attack, x_normal])))
    adv = evaluate(y_eval, model.predict(np.concatenate([x_adv, x_normal])))
    print(f"==== {name}\n  clean:       {clean}\n  adversarial: {adv}")

# Apollon

In [ ]:
# Apollon is implemented in MAB/apollon_mab.py. The class previously defined here trained
# every arm on the whole training set, scored arm k only on samples with label k and never
# updated its Beta posteriors (see the module docstring).

In [ ]:
# Train Apollon: pool of RF, DT, NB, LR and MLP, 2 clusters, Thompson sampling per cluster
mab = ApollonMAB(n_clusters=2, random_state=seed)
mab.fit(x_train, y_train)
print("Arm selection frequencies (RF, DT, NB, LR, MLP):", mab.selection_frequencies(x_test))

In [ ]:
x_eval_clean = np.concatenate([x_attack, x_normal])
print("Apollon, clean data:", evaluate(y_eval, mab.predict(x_eval_clean)))

# 1) Transfer: examples crafted against each single model, replayed on Apollon.
#    This is what the previous version of this notebook measured.
transfer = {}
for name, x_adv in adv_by_model.items():
    transfer[name] = evaluate(y_eval, mab.predict(np.concatenate([x_adv, x_normal])))
    print(f"Transfer from {name}: {transfer[name]}")
print("Mean over source models:",
      {k: float(np.mean([m[k] for m in transfer.values()])) for k in next(iter(transfer.values()))})

# 2) Direct: the attacker queries Apollon itself; every query goes through cluster assignment
#    and Thompson-sampled arm selection, as a request reaching the deployed IDS.
x_adv_direct = run_attack(ATTACK, apollon_classifier(mab, x_attack.shape[1]), x_attack,
                          max_iter=MAX_ITER)
x_adv_direct = restore_functional(x_adv_direct, x_attack, FUNCTIONAL_IDX)
print("Direct attack on Apollon:",
      evaluate(y_eval, mab.predict(np.concatenate([x_adv_direct, x_normal]))))